In [ ]:
import os
import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import csv
import unicodedata

base_url = "https://pennathletics.com"

# Map display names to Penn Athletics URL slugs
TEAM_SLUGS = {
    "Football":            "football",
    "Sprint Football":     "sprint-football",
    "Baseball":            "baseball",
    "Softball":            "softball",
    "Wrestling":           "wrestling",
    "Field Hockey":        "field-hockey",
    "Volleyball":          "womens-volleyball",
    "Womens Basketball":   "womens-basketball",
    "Mens Golf":           "mens-golf",
    "Womens Golf":         "womens-golf",
    "Mens Lacrosse":       "mens-lacrosse",
    "Womens Lacrosse":     "womens-lacrosse",
    "Mens Soccer":         "mens-soccer",
    "Mens Squash":         "mens-squash",
    "Womens Squash":       "womens-squash",
    "Mens Swim":           "mens-swimming-and-diving",
    "Womens Swim":         "womens-swimming-and-diving",
    "Mens Tennis":         "mens-tennis",
    "Womens Tennis":       "womens-tennis",
    "Mens Light Crew":     "mens-crew",
    "Mens Heavy Crew":     "mens-rowing",
    "Womens Crew":         "womens-rowing",
}

YEARS = range(2025, 2027)
output_base_dir = "penn_all_teams"
os.makedirs(output_base_dir, exist_ok=True)

In [6]:
# Helper functions

def clean_display_name(first, last):
    """Title case, no punctuation except hyphens, space-separated: 'First Last'."""
    def clean_part(s):
        s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
        s = re.sub(r"[^a-zA-Z\-]", "", s)
        return s.title()
    return f"{clean_part(first)} {clean_part(last)}"

def clean_filename(first, last):
    """Lowercase, no punctuation except hyphens, underscore-separated: 'first_last.jpg'."""
    def clean_part(s):
        s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
        s = re.sub(r"[^a-zA-Z\-]", "", s).lower()
        return s
    return f"{clean_part(first)}_{clean_part(last)}.jpg"

def clean_key(name):
    name = name.strip().lower()
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("utf-8")
    return re.sub(r"[^\w\s\-]", "", name)

def calculate_grad_year(academic_year, roster_year):
    ay = academic_year.lower().strip()
    if 'freshman' in ay or ay == 'fr':
        return roster_year + 3
    elif 'sophomore' in ay or ay == 'so':
        return roster_year + 2
    elif 'junior' in ay or ay == 'jr':
        return roster_year + 1
    elif 'senior' in ay or ay == 'sr':
        return roster_year
    else:
        return None

In [7]:
# Scrape rosters and download headshots for all teams

headshots_dir = os.path.join(output_base_dir, "headshots")
os.makedirs(headshots_dir, exist_ok=True)

# Keyed by (sport, first, last) — later years overwrite earlier ones
players = {}

for team_name, slug in TEAM_SLUGS.items():
    print(f"\n{'='*50}")
    print(f"Sport: {team_name}  (slug: {slug})")

    for year in YEARS:
        url = f"{base_url}/sports/{slug}/roster/{year}?view=3"
        res = requests.get(url)
        if res.status_code != 200:
            print(f"  ⚠️  {year}: HTTP {res.status_code} — check slug '{slug}'")
            continue

        soup = BeautifulSoup(res.text, "html.parser")
        cards = soup.select("li.sidearm-list-card-item")
        print(f"  📅 {year}: {len(cards)} cards found")

        for card in cards:
            first_el = card.select_one("span.sidearm-roster-player-first-name")
            last_el  = card.select_one("span.sidearm-roster-player-last-name")
            link_el  = card.select_one("a.sidearm-roster-player-name")
            if not (first_el and last_el):
                continue

            first_name = first_el.text.strip()
            last_name  = last_el.text.strip()

            jersey = card.select_one("div.sidearm-roster-player-jersey span")
            jersey_text = jersey.text.strip() if jersey else ""
            if not jersey_text:
                continue

            acad_year = card.select_one("span.sidearm-roster-player-academic-year")
            year_text = acad_year.text.strip() if acad_year else ""

            key = (team_name, clean_key(first_name), clean_key(last_name))
            players[key] = {
                "name":      clean_display_name(first_name, last_name),
                "number":    jersey_text,
                "sport":     team_name,
                "grad_year": calculate_grad_year(year_text, year),
                "_first":    first_name,
                "_last":     last_name,
                "_profile":  urljoin(base_url, link_el["href"]) if link_el else None,
            }

# Download headshots (skip if already saved)
print(f"\n📸 Downloading headshots for {len(players)} players...")
for i, player in enumerate(players.values(), 1):
    filename = clean_filename(player["_first"], player["_last"])
    img_path = os.path.join(headshots_dir, filename)

    if os.path.exists(img_path):
        continue  # already downloaded

    if not player["_profile"]:
        continue

    profile_res = requests.get(player["_profile"])
    if profile_res.status_code != 200:
        continue

    profile_soup = BeautifulSoup(profile_res.text, "html.parser")
    img_tag = profile_soup.select_one("div.sidearm-roster-player-image img")
    if not img_tag or not img_tag.get("src"):
        continue

    img_url = urljoin(base_url, img_tag["src"].split("?")[0])
    try:
        img_data = requests.get(img_url).content
        with open(img_path, "wb") as f:
            f.write(img_data)
        if i % 50 == 0:
            print(f"  ... {i}/{len(players)}")
    except Exception as e:
        print(f"  ❌ {filename}: {e}")

# Save combined CSV
csv_path = os.path.join(output_base_dir, "roster_all_teams.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "number", "sport", "grad_year"])
    writer.writeheader()
    for player in players.values():
        writer.writerow({k: player[k] for k in ["name", "number", "sport", "grad_year"]})

print(f"\n✅ CSV saved to {csv_path} with {len(players)} players.")
print(f"✅ Headshots saved to {headshots_dir}/")


Sport: Football  (slug: football)
  📅 2025: 128 cards found
  📅 2026: 97 cards found

Sport: Sprint Football  (slug: sprint-football)
  📅 2025: 60 cards found
  📅 2026: 52 cards found

Sport: Baseball  (slug: baseball)
  📅 2025: 48 cards found
  📅 2026: 46 cards found

Sport: Softball  (slug: softball)
  📅 2025: 28 cards found
  📅 2026: 29 cards found

Sport: Wrestling  (slug: wrestling)
  📅 2025: 53 cards found
  📅 2026: 53 cards found

Sport: Field Hockey  (slug: field-hockey)
  📅 2025: 34 cards found
  📅 2026: 34 cards found

Sport: Volleyball  (slug: womens-volleyball)
  📅 2025: 28 cards found
  📅 2026: 28 cards found

Sport: Womens Basketball  (slug: womens-basketball)
  📅 2025: 32 cards found
  📅 2026: 32 cards found

Sport: Mens Golf  (slug: mens-golf)
  📅 2025: 15 cards found
  📅 2026: 15 cards found

Sport: Womens Golf  (slug: womens-golf)
  📅 2025: 17 cards found
  📅 2026: 17 cards found

Sport: Mens Lacrosse  (slug: mens-lacrosse)
  📅 2025: 66 cards found
  📅 2026: 58 cards